# Choice of Representation Layers

Thesis: *Experimental Setup*, section *Quantifying Representational Entanglement*, subsections *Choice of the Measurement Space*, *The Entanglement Metric* and *Choice of Representation Layers*.

The concept-word probe comes from `src/generation/run_concept_word_probe.py`.

**Requirements**
- Environment: `easy-language` (`environment-easy-language.yml`)
- Packages: torch (CPU suffices), safetensors, huggingface_hub
- Data: `data/concept_word_probe/`, `data/main_run/image_manifest.csv`, `data/main_run/mivolo_validity.csv` (in this repository)
- Network: the representations of the 720 calibration images from `maxrs/flux2-gal-gsl-tensors` at a pinned revision, about 220 MB.
- GPU: not needed

**Writes:** `exports/concept_word_separation.csv`, `exports/concept_word_decoding.csv`, `exports/gender_direction_validation.csv`, `exports/gender_direction_alignment.csv`, `exports/gender_direction_scene_gaps.csv`

In [1]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "corpus").is_dir())
os.chdir(ROOT)
for folder in ("measurement",):
    sys.path.insert(0, str(ROOT / "src" / folder))

import re
from collections import defaultdict

from gender_direction import (
    MIN_ANCHORS, ROLE_FEMALE, ROLE_MALE, anchor_means, direction_of, load_calibration, reps_loader,
    write_csv,
)
import pandas as pd
import torch
from IPython.display import display

LANGUAGE_FORM = {"ls": "GAL", "sd": "GSL"}

## Choice of the Measurement Space

A direction built from two concept words separates the images of a man from those of a woman at none of the 25 blocks, for männlich/weiblich as for Mann/Frau. As text queries the two words of a pair lie closer together than at the output of the text encoder. The depicted gender is nevertheless linearly decodable from the same representations once they are centred on their scene, at single-stream block 4 in all twelve images.

In [2]:
PROBE_DIR = Path("data/concept_word_probe")
OUT_DIR = Path("exports")
BLOCKS = [f"dual_{i}" for i in range(5)] + [f"single_{i}" for i in range(20)]
PAIR_LABEL = {("männlich", "weiblich"): "männlich/weiblich", ("Mann", "Frau"): "Mann/Frau"}

In [3]:
word_cosine = pd.read_csv(PROBE_DIR / "concept_word_cosine.csv")
word_cosine["concept_pair"] = [PAIR_LABEL[(a, b)] for a, b in zip(word_cosine["word_a"], word_cosine["word_b"])]
word_cosine["tokens"] = word_cosine["token_a"] + " / " + word_cosine["token_b"]
print("Cosine of the two concept words, token by token")
for label, group in word_cosine.groupby("concept_pair", sort=False):
    display(group.pivot_table(index="stage", columns="tokens", values="cosine", sort=False).round(2)
            .rename_axis(index=label, columns=None))

Cosine of the two concept words, token by token


,m / we,Ã¤nn / ib,lich / lich
männlich/weiblich,,,
encoder,0.76,0.42,0.94
context_embedder,0.89,0.56,0.96
dual_0,0.99,0.89,0.99
dual_1,0.95,0.69,0.98
dual_2,0.95,0.71,0.98
dual_3,0.94,0.70,0.98
dual_4,0.94,0.73,0.98


,M / Fra,ann / u
Mann/Frau,,
encoder,0.61,0.41
context_embedder,0.82,0.54
dual_0,0.98,0.89
dual_1,0.90,0.71
dual_2,0.90,0.68
dual_3,0.89,0.72
dual_4,0.89,0.73


In [4]:
scores = pd.read_csv(PROBE_DIR / "concept_probe_scores.csv")
scores["block"] = scores["stream"] + "_" + scores["layer_idx"].astype(str)
scores["concept_pair"] = [PAIR_LABEL[(a, b)] for a, b in zip(scores["word_a"], scores["word_b"])]
wide = scores.pivot_table(index=["concept_pair", "block", "pair_idx", "seed"], columns="role",
                          values="global_diff_score").dropna()
wide["gap"] = wide["male"] - wide["female"]
separation = (wide.groupby(["concept_pair", "block"])["gap"]
              .agg(man_higher=lambda g: int((g > 0).sum()), n="size", mean_gap="mean").reset_index())
separation.to_csv(OUT_DIR / "concept_word_separation.csv", index=False, lineterminator="\r\n")

print("Seed-matched comparisons in which the image of the man scores higher")
display(separation.assign(share=separation["man_higher"].astype(str) + "/" + separation["n"].astype(str))
        .pivot(index="block", columns="concept_pair", values="share")
        .reindex(index=BLOCKS, columns=list(PAIR_LABEL.values()))
        .rename_axis(columns=None))
separated = separation[separation["man_higher"] == separation["n"]]
print("blocks that separate in all comparisons:",
      ", ".join(separated["concept_pair"] + " " + separated["block"]) or "none")

Seed-matched comparisons in which the image of the man scores higher


,männlich/weiblich,Mann/Frau
block,,
dual_0,3/6,2/6
dual_1,3/6,3/6
dual_2,1/6,4/6
dual_3,5/6,3/6
dual_4,2/6,3/6
single_0,4/6,1/6
single_1,1/6,2/6
single_2,1/6,2/6
single_3,2/6,1/6


blocks that separate in all comparisons: none


In [5]:
def load_reps(reps_dir: Path) -> list[dict]:
    items = []
    for path in sorted(reps_dir.glob("reps_pair*_*_seed*.pt")):
        m = re.match(r"reps_pair(\d+)_(male|female)_seed(\d+)\.pt", path.name)
        reps = torch.load(path, map_location="cpu", weights_only=True)
        items.append({"pair": int(m.group(1)), "role": m.group(2), "seed": int(m.group(3)),
                      "reps": {k: v.squeeze(0) for k, v in reps.items()}})
    return items


def loo_accuracy(items: list[dict], layer: str, scene_centered: bool) -> tuple[int, int, float]:
    """Leave-one-out with the male-minus-female mean direction; the margin is positive on the correct side."""
    correct, total, margins = 0, 0, []
    for i, test in enumerate(items):
        train = [it for j, it in enumerate(items) if j != i]
        if scene_centered:
            scene_means = {}
            for pair in {it["pair"] for it in train}:
                scene_reps = torch.stack([it["reps"][layer] for it in train if it["pair"] == pair])
                scene_means[pair] = scene_reps.mean(dim=0)
            get = lambda it: it["reps"][layer] - scene_means[it["pair"]]
        else:
            grand_mean = torch.stack([it["reps"][layer] for it in train]).mean(dim=0)
            get = lambda it: it["reps"][layer] - grand_mean
        male = torch.stack([get(it) for it in train if it["role"] == "male"]).mean(dim=0)
        female = torch.stack([get(it) for it in train if it["role"] == "female"]).mean(dim=0)
        margin = float(get(test) @ (male - female))
        is_male = test["role"] == "male"
        correct += int((margin > 0) == is_male)
        total += 1
        margins.append(margin if is_male else -margin)
    return correct, total, sum(margins) / len(margins)

In [6]:
probe_items = load_reps(PROBE_DIR)
rows = []
for block in BLOCKS:
    c_raw, n, m_raw = loo_accuracy(probe_items, block, scene_centered=False)
    c_cen, _, m_cen = loo_accuracy(probe_items, block, scene_centered=True)
    rows.append({"block": block, "raw_correct": c_raw, "centered_correct": c_cen, "n": n,
                 "raw_margin": m_raw, "centered_margin": m_cen})
decoding = pd.DataFrame(rows)
decoding.to_csv(OUT_DIR / "concept_word_decoding.csv", index=False, lineterminator="\r\n")

print(f"Depicted gender decoded leave-one-out from the {n} mean representations")
display(decoding.set_index("block").round(2))

Depicted gender decoded leave-one-out from the 12 mean representations


,raw_correct,centered_correct,n,raw_margin,centered_margin
block,,,,,
dual_0,0,2,12,-36.99,-4.35
dual_1,0,0,12,-51.28,-10.53
dual_2,0,0,12,-56.35,-8.33
dual_3,0,1,12,-60.00,-16.02
dual_4,0,2,12,-371.36,-66.81
single_0,0,6,12,-1.26,-0.11
single_1,0,6,12,-1.56,-0.23
single_2,0,7,12,-1.97,-0.09
single_3,0,9,12,-2.74,0.36


## The gate

Validation of the gender direction on held-out scenes: per single-stream block, the share of seed-matched anchor pairs the direction orders correctly (leave-one-scene-out, gate 0.80 in both language forms, fixed before the numbers existed), and the cosine between the directions of the two language forms. All twenty blocks pass; block 4 has the highest cosine and is the primary block, block 5 the replication.

In [7]:
THRESHOLD = 0.80


def evaluate_fold(train: list[dict], test: list[dict], layer: str) -> list[float]:
    """Margin in anchor units, male minus female, for every seed-matched pair held out."""
    g = direction_of(train, layer)
    if g is None:
        return []
    denom = float(g @ g)
    if denom == 0:
        return []
    by_pair: dict[tuple, dict] = defaultdict(dict)
    for it in test:
        by_pair[(it["profession"], it["scene"], it["seed"])][it["role"]] = it["reps"][layer]
    margins = []
    for reps in by_pair.values():
        if ROLE_MALE in reps and ROLE_FEMALE in reps:
            margins.append(float((reps[ROLE_MALE] - reps[ROLE_FEMALE]) @ g) / denom)
    return margins


def run_split(items: list[dict], layer: str, by: str) -> tuple[int, int, float]:
    """by='scene' holds out one profession-scene at a time, by='profession' one profession."""
    keys = sorted({(it["profession"], it["scene"]) if by == "scene" else (it["profession"],)
                   for it in items})
    margins: list[float] = []
    for key in keys:
        def in_fold(it, key=key):
            return ((it["profession"], it["scene"]) == key if by == "scene"
                    else (it["profession"],) == key)
        test = [it for it in items if in_fold(it)]
        train = [it for it in items if not in_fold(it)]
        margins.extend(evaluate_fold(train, test, layer))
    correct = sum(1 for m in margins if m > 0)
    mean_margin = sum(margins) / len(margins) if margins else float("nan")
    return correct, len(margins), mean_margin

In [8]:
RUN_DIR = Path("data/main_run")
LAYER_PREFIX = "single"
PERSON_FILTER = RUN_DIR / "mivolo_validity.csv"  # None estimates on every calibration image

In [9]:
items = load_calibration(RUN_DIR, PERSON_FILTER, load_reps=reps_loader("reps/*_explicit_*.safetensors"))
varieties = sorted({it["variety"] for it in items})
layers = sorted((k for k in items[0]["reps"] if k.startswith(LAYER_PREFIX + "_")),
                key=lambda k: int(k.split("_")[1]))
print(f"{len(items)} calibration images, language forms {varieties}, {len(layers)} blocks, "
      f"at least {MIN_ANCHORS} anchors per pole for a scene to enter the direction")

Fetching ... files: 0it [00:00, ?it/s]

single-person filter: 605 of 720 calibration images kept


605 calibration images, language forms ['ls', 'sd'], 20 blocks, at least 3 anchors per pole for a scene to enter the direction


In [10]:
rows, per_layer = [], defaultdict(dict)
for layer in layers:
    for variety in varieties:
        subset = [it for it in items if it["variety"] == variety]
        for by in ("scene", "profession"):
            correct, n, margin = run_split(subset, layer, by)
            acc = correct / n if n else float("nan")
            rows.append({"layer": layer, "variety": variety, "split": f"leave_one_{by}_out",
                         "n_pairs": n, "n_correct": correct, "accuracy": round(acc, 4),
                         "mean_margin": round(margin, 4)})
            if by == "scene":
                per_layer[layer][variety] = acc

align, gaps, cosine = [], [], {}
for layer in layers:
    dirs = {}
    for v in varieties:
        subset = [it for it in items if it["variety"] == v]
        g = direction_of(subset, layer)
        dirs[v] = g
        if g is None:
            continue
        denom = float(g @ g)
        for (prof, scene), (m, f, nm, nf) in sorted(anchor_means(subset, layer).items()):
            gaps.append({"layer": layer, "variety": v, "profession": prof, "scene": scene,
                         "n_male": nm, "n_female": nf,
                         "gap": round(float((m - f) @ g) / denom, 4)})
    if len(varieties) == 2 and all(dirs[v] is not None for v in varieties):
        a, b = (dirs[v] for v in varieties)
        cos = float(a @ b / (a.norm() * b.norm())) if a.norm() * b.norm() > 0 else 0.0
        cosine[layer] = cos
        align.append({"layer": layer, "varieties": "_vs_".join(varieties),
                      "cosine": round(cos, 4)})

OUT_DIR.mkdir(parents=True, exist_ok=True)
write_csv(OUT_DIR / "gender_direction_validation.csv", rows)
write_csv(OUT_DIR / "gender_direction_alignment.csv", align)
write_csv(OUT_DIR / "gender_direction_scene_gaps.csv", gaps)

In [11]:
# The gate, applied mechanically so that the threshold cannot drift after the fact.
print(f"\ngate: a block passes at accuracy >= {THRESHOLD:.2f} in every language form "
      f"(leave-one-scene-out)")
passing, gate = [], []
for layer in layers:
    accs = per_layer[layer]
    ok = all(a >= THRESHOLD for a in accs.values()) and len(accs) == len(varieties)
    gate.append({"block": layer, **{LANGUAGE_FORM[v]: round(accs[v], 3) for v in varieties},
                 "cosine": round(cosine[layer], 3) if layer in cosine else None, "passes": ok})
    if ok and layer in cosine:
        passing.append((cosine[layer], layer))
display(pd.DataFrame(gate).set_index("block"))
if not passing:
    print("\n  no block passes in both language forms")
else:
    passing.sort(reverse=True)
    primary = passing[0][1]
    print(f"\n  primary block (highest cosine among passing): {primary} "
          f"(cos {passing[0][0]:.3f})")
    if len(passing) > 1:
        print(f"  replication block:                            {passing[1][1]} "
              f"(cos {passing[1][0]:.3f})")
    for v in varieties:
        vals = sorted(r["gap"] for r in gaps if r["layer"] == primary and r["variety"] == v)
        if vals:
            print(f"  within-scene anchor gap at {primary}, {LANGUAGE_FORM[v]}: {len(vals)} scenes, "
                  f"min {vals[0]:.3f}, median {vals[len(vals) // 2]:.3f}, max {vals[-1]:.3f}")
print(f"\nwritten: gender_direction_validation.csv, gender_direction_alignment.csv, "
      f"gender_direction_scene_gaps.csv in {OUT_DIR}")


gate: a block passes at accuracy >= 0.80 in every language form (leave-one-scene-out)


,GAL,GSL,cosine,passes
block,,,,
single_0,1.000,1.000,0.918,True
single_1,1.000,0.978,0.921,True
single_2,1.000,1.000,0.948,True
single_3,1.000,1.000,0.969,True
single_4,1.000,1.000,0.986,True
single_5,1.000,1.000,0.980,True
single_6,1.000,1.000,0.977,True
single_7,1.000,1.000,0.962,True
single_8,0.993,1.000,0.949,True



  primary block (highest cosine among passing): single_4 (cos 0.986)
  replication block:                            single_5 (cos 0.980)
  within-scene anchor gap at single_4, GAL: 32 scenes, min 0.553, median 0.954, max 1.530
  within-scene anchor gap at single_4, GSL: 32 scenes, min 0.562, median 1.032, max 1.631

written: gender_direction_validation.csv, gender_direction_alignment.csv, gender_direction_scene_gaps.csv in exports
